# IMPORTS

In [1]:
import pandas as pd
import numpy as np 

In [2]:
shootings = pd.read_csv(r"C:\Users\arime\Downloads\Shootings_(2006-Present)_20260920.csv")
shootings.shape

(24310, 13)

# SHOOTINGS DF
#### INCIDENT_KEY: unique key for each incident. Used as PK to join to victims + offenders 
#### OCCUR_DATE: date of shooting incident, changed to datetime format 
#### OCCUR_TIME: time of shooting incident, changed to timedelta format 
#### TIME_OF_DAY: OCCUR_TIME broken into categories of time of day
#### BORO: borough where incident occured. 
#### BORO_TOTAL: borough where incident occured. PBXN + PBXS (Patrol Bronx North + South) combined with BRONX.
#### LOC_OF_OCCUR_DESC: flag indicating where incident occured. INSIDE or OUTSIDE 
#### PRECINCT: precinct where shooting incident occured
#### JURISDICTION_CODE: jurisdiction where shooting incident occured (0-Patrol, 1-Transit, 2-Housing, 3+ represent non NYPD jurisdictions) - 2 nulls
#### LOC_CLASSFCTN_DESC: category describing type of location 
#### LOCATION_DESC: category for specific type of location of shooting incident 
#### X_COORD_CD: x coordinates 
#### Y_COORD_CD: y coordinates 
#### Latitude: latitude coordinates 
#### Longitude: longitude coordinates 

In [3]:
shootings.head(5)

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
0,212200405,04/20/2020,19:36:00,BROOKLYN,OUTSIDE,79,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1000301,192923,-73.942118,40.696199
1,139554478,11/07/2014,03:20:00,BRONX,INSIDE,42,0.0,DWELLING,PVT HOUSE,1012185,241494,-73.899059,40.829485
2,167619814,08/01/2017,00:35:00,BRONX,OUTSIDE,47,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1030360,261043,-73.833250,40.883065
3,212516311,04/29/2020,15:30:00,BRONX,OUTSIDE,46,0.0,STREET,NaN,1009085,248078,-73.910236,40.847565
4,270150948,06/20/2023,23:41:00,QUEENS,OUTSIDE,108,0.0,STREET,NaN,1004925,209948,-73.925390,40.742919


In [4]:
shootings.isnull().sum()

INCIDENT_KEY              0
OCCUR_DATE                0
OCCUR_TIME                0
BORO                      0
LOC_OF_OCCUR_DESC         0
PRECINCT                  0
JURISDICTION_CODE         2
LOC_CLASSFCTN_DESC       32
LOCATION_DESC         14344
X_COORD_CD                0
Y_COORD_CD                0
Latitude                  0
Longitude                 0
dtype: int64

In [5]:
shootings.dtypes

INCIDENT_KEY            int64
OCCUR_DATE             object
OCCUR_TIME             object
BORO                   object
LOC_OF_OCCUR_DESC      object
PRECINCT                int64
JURISDICTION_CODE     float64
LOC_CLASSFCTN_DESC     object
LOCATION_DESC          object
X_COORD_CD              int64
Y_COORD_CD              int64
Latitude              float64
Longitude             float64
dtype: object

### Check INCIDENT_KEY has no duplicates

In [6]:
print(shootings['INCIDENT_KEY'].duplicated().any())

False


### Change OCCUR_DATE and OCCUR_TIME to datetime/timedelta

In [7]:
shootings['OCCUR_DATE'] = pd.to_datetime(shootings['OCCUR_DATE'], format='%m/%d/%Y', errors='coerce')
#shootings['OCCUR_TIME'] = pd.to_timedelta(shootings['OCCUR_TIME']) - ERROR, some messed up values 

In [8]:
#copy of shootings df to find incorrect data
time_nat = shootings.copy()

#get indices of messed up values
time_nat['OCCUR_TIME'] = pd.to_timedelta(time_nat['OCCUR_TIME'], errors='coerce')
nat_indices = time_nat[time_nat['OCCUR_TIME'].isna()].index

nat_rows = shootings.loc[nat_indices, 'OCCUR_TIME']
print(nat_rows)

1732     1899-12-31T17:51:00.000
1733     1899-12-31T16:45:00.000
1734     1899-12-31T05:00:00.000
1735     1899-12-31T00:55:00.000
1736     1899-12-31T06:20:00.000
                  ...           
24305    1899-12-31T22:28:00.000
24306    1899-12-31T19:31:00.000
24307    1899-12-31T17:06:00.000
24308    1899-12-31T19:16:00.000
24309    1899-12-31T22:22:00.000
Name: OCCUR_TIME, Length: 322, dtype: object


In [9]:
# check if all times have the same format
print(nat_rows.str.startswith('1899-12-31T').all())
print(nat_rows.str.endswith('.000').all())

True
True


In [10]:
#clean up rows on main shootings df
shootings['OCCUR_TIME'] = shootings['OCCUR_TIME'].str.replace('1899-12-31T','', regex=False)
shootings['OCCUR_TIME'] = shootings['OCCUR_TIME'].str.replace('.000','', regex=False)
shootings['OCCUR_TIME'] = pd.to_timedelta(shootings['OCCUR_TIME'])

### Create TIME_OF_DAY 

In [11]:
hours = shootings['OCCUR_TIME'].dt.components.hours

#bins + labels for time of day 
bins = [0, 6, 12, 17, 21, 24]
labels = ['NIGHT', 'MORNING', 'AFTERNOON', 'EVENING', 'NIGHT']

shootings['TIME_OF_DAY'] = pd.cut(hours, bins=bins, labels=labels, right=False, ordered=False)

print(shootings['TIME_OF_DAY'].value_counts())

TIME_OF_DAY
NIGHT        14100
EVENING       4934
AFTERNOON     3580
MORNING       1696
Name: count, dtype: int64


In [12]:
tod = shootings.pop('TIME_OF_DAY')
shootings.insert(3, 'TIME_OF_DAY', tod)
shootings.head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,TIME_OF_DAY,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
0,212200405,2020-04-20,0 days 19:36:00,EVENING,BROOKLYN,OUTSIDE,79,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1000301,192923,-73.942118,40.696199
1,139554478,2014-11-07,0 days 03:20:00,NIGHT,BRONX,INSIDE,42,0.0,DWELLING,PVT HOUSE,1012185,241494,-73.899059,40.829485
2,167619814,2017-08-01,0 days 00:35:00,NIGHT,BRONX,OUTSIDE,47,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1030360,261043,-73.833250,40.883065
3,212516311,2020-04-29,0 days 15:30:00,AFTERNOON,BRONX,OUTSIDE,46,0.0,STREET,NaN,1009085,248078,-73.910236,40.847565
4,270150948,2023-06-20,0 days 23:41:00,NIGHT,QUEENS,OUTSIDE,108,0.0,STREET,NaN,1004925,209948,-73.925390,40.742919


### Inspect categorical columns

In [13]:
#create df of the categorical columns, done manually to inspect the columns that will be relevant for my analysis 
cat_columns = shootings[['TIME_OF_DAY','BORO','LOC_OF_OCCUR_DESC','JURISDICTION_CODE','LOC_CLASSFCTN_DESC','LOCATION_DESC']]

#loop through every column of categorical_columns df, print their unique values and the counts of them
for column in cat_columns: 
    print(f"Column: {column}")
    print(cat_columns[column].value_counts())
    print("-" * 30)

Column: TIME_OF_DAY
TIME_OF_DAY
NIGHT        14100
EVENING       4934
AFTERNOON     3580
MORNING       1696
Name: count, dtype: int64
------------------------------
Column: BORO
BORO
BROOKLYN         9777
BRONX            6925
QUEENS           3612
MANHATTAN        3193
STATEN ISLAND     680
PBXN               63
PBXS               60
Name: count, dtype: int64
------------------------------
Column: LOC_OF_OCCUR_DESC
LOC_OF_OCCUR_DESC
OUTSIDE    20859
INSIDE      3451
Name: count, dtype: int64
------------------------------
Column: JURISDICTION_CODE
JURISDICTION_CODE
0.0    20284
2.0     3955
1.0       69
Name: count, dtype: int64
------------------------------
Column: LOC_CLASSFCTN_DESC
LOC_CLASSFCTN_DESC
STREET         15902
HOUSING         4495
DWELLING        1963
COMMERCIAL      1022
PLAYGROUND       286
VEHICLE          277
OTHER            138
TRANSIT           80
PARKING LOT       77
TAXI/LIVERY       33
(null)             5
Name: count, dtype: int64
----------------------------

### Look at BORO (PBXN + PBXS) - shootings for Bronx ended Dec 27, 2025 and became split into PBXN + PBXS starting Jan 1, 2026. Create new BORO_TOTAL that has a BRONX_TOTAL to account for all Bronx shootings 

In [14]:
shootings[shootings['BORO']=='BRONX'].sort_values(by='OCCUR_DATE', ascending=False).head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,TIME_OF_DAY,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
24192,318049855,2025-12-27,0 days 18:25:00,EVENING,BRONX,INSIDE,49,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1025102,256070,-73.852295,40.869441
23276,318080322,2025-12-27,0 days 05:42:00,NIGHT,BRONX,OUTSIDE,44,0.0,STREET,NaN,1006999,245897,-73.917784,40.841585
22406,317958644,2025-12-25,0 days 02:02:00,NIGHT,BRONX,OUTSIDE,52,0.0,STREET,NaN,1017325,259857,-73.880395,40.879868
23544,317915762,2025-12-23,0 days 12:11:00,AFTERNOON,BRONX,OUTSIDE,47,0.0,STREET,NaN,1024465,261172,-73.854568,40.883448
22817,317774978,2025-12-20,0 days 07:00:00,MORNING,BRONX,OUTSIDE,48,0.0,STREET,NaN,1015734,247131,-73.886208,40.844945


In [15]:
shootings[shootings['BORO']=='PBXN'].sort_values(by='OCCUR_DATE', ascending=True).head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,TIME_OF_DAY,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
1965,318242316,2026-01-01,0 days 06:50:00,MORNING,PBXN,OUTSIDE,46,0.0,STREET,(null),1009343,247443,40.845824,-73.909305
1922,318242319,2026-01-01,0 days 07:57:00,MORNING,PBXN,INSIDE,47,0.0,STREET,(null),1020977,256324,40.870158,-73.867206
1928,318227966,2026-01-01,0 days 01:46:00,NIGHT,PBXN,OUTSIDE,47,0.0,STREET,(null),1020689,256478,40.870581,-73.868246
2014,318242320,2026-01-01,0 days 18:27:00,EVENING,PBXN,INSIDE,47,0.0,STREET,(null),1023850,265667,40.895788,-73.856764
2002,318580566,2026-01-08,0 days 13:45:00,AFTERNOON,PBXN,INSIDE,46,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1005842,248069,40.847551,-73.921956


In [16]:
shootings['BORO_TOTAL'] = shootings['BORO']

bronx_mapping = {
    'BRONX': 'BRONX',
    'PBXN': 'BRONX',
    'PBXS': 'BRONX'
}

shootings['BORO_TOTAL'] = shootings['BORO_TOTAL'].replace(bronx_mapping)

In [17]:
boro_total = shootings.pop('BORO_TOTAL')
shootings.insert(5, 'BORO_TOTAL', boro_total)
shootings.head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,TIME_OF_DAY,BORO,BORO_TOTAL,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
0,212200405,2020-04-20,0 days 19:36:00,EVENING,BROOKLYN,BROOKLYN,OUTSIDE,79,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1000301,192923,-73.942118,40.696199
1,139554478,2014-11-07,0 days 03:20:00,NIGHT,BRONX,BRONX,INSIDE,42,0.0,DWELLING,PVT HOUSE,1012185,241494,-73.899059,40.829485
2,167619814,2017-08-01,0 days 00:35:00,NIGHT,BRONX,BRONX,OUTSIDE,47,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1030360,261043,-73.833250,40.883065
3,212516311,2020-04-29,0 days 15:30:00,AFTERNOON,BRONX,BRONX,OUTSIDE,46,0.0,STREET,NaN,1009085,248078,-73.910236,40.847565
4,270150948,2023-06-20,0 days 23:41:00,NIGHT,QUEENS,QUEENS,OUTSIDE,108,0.0,STREET,NaN,1004925,209948,-73.925390,40.742919


In [18]:
shootings['BORO_TOTAL'].value_counts()

BORO_TOTAL
BROOKLYN         9777
BRONX            7048
QUEENS           3612
MANHATTAN        3193
STATEN ISLAND     680
Name: count, dtype: int64

### Handle LOCATION_DESC and LOC_CLASSFCTN_DESC nulls

In [19]:
shootings['LOCATION_DESC'] = shootings['LOCATION_DESC'].replace(['(null)', 'NONE'], 'UNKNOWN')
shootings['LOCATION_DESC'] = shootings['LOCATION_DESC'].fillna('UNKNOWN')

shootings['LOCATION_DESC'].value_counts()

LOCATION_DESC
UNKNOWN                      14698
MULTI DWELL - PUBLIC HOUS     4377
MULTI DWELL - APT BUILD       2383
PVT HOUSE                      742
GROCERY/BODEGA                 591
BAR/NIGHT CLUB                 489
COMMERCIAL BLDG                228
RESTAURANT/DINER               172
FAST FOOD                       88
BEAUTY/NAIL SALON               84
HOSPITAL                        83
GAS STATION                     61
SOCIAL CLUB/POLICY LOCATI       47
SMALL MERCHANT                  33
STORE UNCLASSIFIED              32
HOTEL/MOTEL                     31
LIQUOR STORE                    28
DRY CLEANER/LAUNDRY             27
SUPERMARKET                     15
CLOTHING BOUTIQUE               12
JEWELRY STORE                   11
VARIETY STORE                   10
CHAIN STORE                      9
CANDY STORE                      9
DRUG STORE                       8
FACTORY/WAREHOUSE                6
DEPT STORE                       6
TELECOMM. STORE                  5
SHOE S

In [20]:
shootings['LOC_CLASSFCTN_DESC'] = shootings['LOC_CLASSFCTN_DESC'].replace('(null)', 'UNKNOWN')
shootings['LOC_CLASSFCTN_DESC'] = shootings['LOC_CLASSFCTN_DESC'].fillna('UNKNOWN')

shootings['LOC_CLASSFCTN_DESC'].value_counts()

LOC_CLASSFCTN_DESC
STREET         15902
HOUSING         4495
DWELLING        1963
COMMERCIAL      1022
PLAYGROUND       286
VEHICLE          277
OTHER            138
TRANSIT           80
PARKING LOT       77
UNKNOWN           37
TAXI/LIVERY       33
Name: count, dtype: int64

In [21]:
shootings.isnull().sum()

INCIDENT_KEY          0
OCCUR_DATE            0
OCCUR_TIME            0
TIME_OF_DAY           0
BORO                  0
BORO_TOTAL            0
LOC_OF_OCCUR_DESC     0
PRECINCT              0
JURISDICTION_CODE     2
LOC_CLASSFCTN_DESC    0
LOCATION_DESC         0
X_COORD_CD            0
Y_COORD_CD            0
Latitude              0
Longitude             0
dtype: int64

### Check coordinates - some latitude and longitudes are flipped and 1 row with no lat + long

In [22]:
shootings[shootings['Latitude'] < 0].head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,TIME_OF_DAY,BORO,BORO_TOTAL,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
0,212200405,2020-04-20,0 days 19:36:00,EVENING,BROOKLYN,BROOKLYN,OUTSIDE,79,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1000301,192923,-73.942118,40.696199
1,139554478,2014-11-07,0 days 03:20:00,NIGHT,BRONX,BRONX,INSIDE,42,0.0,DWELLING,PVT HOUSE,1012185,241494,-73.899059,40.829485
2,167619814,2017-08-01,0 days 00:35:00,NIGHT,BRONX,BRONX,OUTSIDE,47,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1030360,261043,-73.833250,40.883065
3,212516311,2020-04-29,0 days 15:30:00,AFTERNOON,BRONX,BRONX,OUTSIDE,46,0.0,STREET,UNKNOWN,1009085,248078,-73.910236,40.847565
4,270150948,2023-06-20,0 days 23:41:00,NIGHT,QUEENS,QUEENS,OUTSIDE,108,0.0,STREET,UNKNOWN,1004925,209948,-73.925390,40.742919


In [23]:
flipped_rows = (shootings['Latitude'] < 0) & (shootings['Longitude'] > 0)

shootings.loc[flipped_rows, ['Latitude', 'Longitude']] = shootings.loc[flipped_rows, ['Longitude', 'Latitude']].values

shootings[shootings['Latitude'] < 0].head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,TIME_OF_DAY,BORO,BORO_TOTAL,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude


In [24]:
print(shootings['Latitude'].describe())
print(shootings['Longitude'].describe())

count    24310.000000
mean        40.736138
std          0.275517
min          0.000000
25%         40.668293
50%         40.700351
75%         40.823849
max         40.910816
Name: Latitude, dtype: float64
count    24310.000000
mean       -73.906043
std          0.478579
min        -74.249755
25%        -73.942722
50%        -73.915308
75%        -73.883190
max          0.000000
Name: Longitude, dtype: float64


In [25]:
shootings[shootings['Latitude'] == 0.0]

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,TIME_OF_DAY,BORO,BORO_TOTAL,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
1858,325386857,2026-05-20,0 days 05:50:00,NIGHT,PBXS,BRONX,OUTSIDE,40,0.0,STREET,UNKNOWN,1005028,234516,0.0,0.0


### Order oldest to most recent

In [26]:
shootings = shootings.sort_values(by=['OCCUR_DATE', 'OCCUR_TIME'])

In [27]:
shootings.head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,TIME_OF_DAY,BORO,BORO_TOTAL,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
1457,9953245,2006-01-01,0 days 02:00:00,NIGHT,BRONX,BRONX,OUTSIDE,48,0.0,STREET,UNKNOWN,1013404,251800,40.857768,-73.894610
1637,9953252,2006-01-01,0 days 02:22:00,NIGHT,MANHATTAN,MANHATTAN,OUTSIDE,28,0.0,STREET,UNKNOWN,998811,233547,40.807705,-73.947403
1458,9953250,2006-01-01,0 days 02:34:00,NIGHT,QUEENS,QUEENS,OUTSIDE,114,0.0,COMMERCIAL,BAR/NIGHT CLUB,1002576,221583,40.774860,-73.933835
552,9953247,2006-01-01,0 days 03:30:00,NIGHT,BROOKLYN,BROOKLYN,OUTSIDE,67,0.0,STREET,UNKNOWN,999316,176460,40.651014,-73.945707
4668,9953246,2006-01-01,0 days 05:51:00,NIGHT,BRONX,BRONX,OUTSIDE,44,0.0,STREET,UNKNOWN,1007446,243897,40.836094,-73.916175


# VICTIMS DF
#### - INCIDENT_KEY: unique ID for each incident, PK across dfs 
#### - VICTIM_ID: unique ID for each victim 
#### - VICTIM_AGE_GROUP: victim's age group 
#### - VICIM_SEX: victim's sex description
#### - VICTIM_RACE: victim's race description 
#### - STAT_MURDER_FLG: indicator if shooting resulted in victim's death 

In [28]:
victims = pd.read_csv(r"C:\Users\arime\Downloads\Shooting_Victims_(2006-Present)_20260920.csv")
victims.shape

(29293, 6)

In [29]:
victims.head()

,INCIDENT_KEY,VICTIM_ID,VICTIM_AGE_GROUP,VICTIM_SEX,VICTIM_RACE,STAT_MURDER_FLG
0,299453927,299453927-1,25-44,MALE,BLACK,N
1,212336415,212336415-1,18-24,MALE,BLACK,N
2,280768895,280768895-1,25-44,MALE,BLACK,N
3,141824677,141824677-27687,25-44,MALE,BLACK HISPANIC,N
4,270079203,270079203-1,45-64,MALE,BLACK,Y


In [30]:
victims.dtypes

INCIDENT_KEY         int64
VICTIM_ID           object
VICTIM_AGE_GROUP    object
VICTIM_SEX          object
VICTIM_RACE         object
STAT_MURDER_FLG     object
dtype: object

### Check for nulls

In [31]:
victims.isnull().sum()

INCIDENT_KEY        0
VICTIM_ID           1
VICTIM_AGE_GROUP    1
VICTIM_SEX          1
VICTIM_RACE         1
STAT_MURDER_FLG     1
dtype: int64

### Drop row with all nulls

In [32]:
victims[victims['VICTIM_ID'].isnull()]

,INCIDENT_KEY,VICTIM_ID,VICTIM_AGE_GROUP,VICTIM_SEX,VICTIM_RACE,STAT_MURDER_FLG
28911,203259656,NaN,NaN,NaN,NaN,NaN


In [33]:
victims = victims.dropna(subset=['VICTIM_ID'])
victims.isnull().sum()

INCIDENT_KEY        0
VICTIM_ID           0
VICTIM_AGE_GROUP    0
VICTIM_SEX          0
VICTIM_RACE         0
STAT_MURDER_FLG     0
dtype: int64

### Check if INCIDENT_KEY is unique - multiple victims per some incidents. Check if all victim INCIDENT_KEY are in shootings 

In [34]:
print(victims['INCIDENT_KEY'].duplicated().any())

True


In [35]:
victims['INCIDENT_KEY'].value_counts()

INCIDENT_KEY
311327863    14
200331333    12
80484245     11
298756519    10
243566417    10
             ..
9953248       1
212640532     1
139621460     1
305930898     1
289881880     1
Name: count, Length: 24311, dtype: int64

In [36]:
victims[victims['INCIDENT_KEY']==311327863]

,INCIDENT_KEY,VICTIM_ID,VICTIM_AGE_GROUP,VICTIM_SEX,VICTIM_RACE,STAT_MURDER_FLG
21357,311327863,311327863-1,25-44,MALE,BLACK,Y
21358,311327863,311327863-2,25-44,MALE,BLACK,Y
21359,311327863,311327863-3,18-24,MALE,BLACK,Y
21360,311327863,311327863-4,25-44,MALE,BLACK,N
21361,311327863,311327863-5,25-44,FEMALE,BLACK,N
21362,311327863,311327863-6,25-44,FEMALE,BLACK,N
21363,311327863,311327863-7,25-44,FEMALE,BLACK,N
21364,311327863,311327863-8,25-44,MALE,BLACK,N
21365,311327863,311327863-9,25-44,MALE,BLACK,N
21366,311327863,311327863-10,25-44,MALE,BLACK,N


In [37]:
victims[~victims['INCIDENT_KEY'].isin(shootings['INCIDENT_KEY'])]

,INCIDENT_KEY,VICTIM_ID,VICTIM_AGE_GROUP,VICTIM_SEX,VICTIM_RACE,STAT_MURDER_FLG
11568,78019135,78019135-20239,25-44,MALE,BLACK HISPANIC,N


### Inspect categorical columns

In [38]:
#create df of the categorical columns, done manually to inspect the columns that will be relevant for my analysis 
categorical_columns = victims[['VICTIM_ID','VICTIM_AGE_GROUP','VICTIM_SEX','VICTIM_RACE','STAT_MURDER_FLG']]

#loop through every column of categorical_columns df, print their unique values and the counts of them
for column in categorical_columns: 
    print(f"Column: {column}")
    print(categorical_columns[column].value_counts())
    print("-" * 30)

Column: VICTIM_ID
VICTIM_ID
318646005-1      2
318780432-1      2
319347278-1      2
319963895-1      2
319963895-2      2
                ..
10187502-8267    1
10187517-8285    1
10240242-8294    1
10333320-8333    1
327260762-1      1
Name: count, Length: 29128, dtype: int64
------------------------------
Column: VICTIM_AGE_GROUP
VICTIM_AGE_GROUP
25-44      13453
18-24      10529
<18         3037
45-64       1997
65+          216
UNKNOWN       59
1022           1
Name: count, dtype: int64
------------------------------
Column: VICTIM_SEX
VICTIM_SEX
MALE        26601
FEMALE       2679
UNKNOWN        11
INTERSEX        1
Name: count, dtype: int64
------------------------------
Column: VICTIM_RACE
VICTIM_RACE
BLACK                             21156
WHITE HISPANIC                     4168
BLACK HISPANIC                     2752
WHITE                               703
ASIAN / PACIFIC ISLANDER            436
UNKNOWN                              64
AMERICAN INDIAN/ALASKAN NATIVE       13
Na

### Drop VICTIM_ID duplicates

In [39]:
victims[victims.duplicated(subset=['INCIDENT_KEY', 'VICTIM_ID'])]

,INCIDENT_KEY,VICTIM_ID,VICTIM_AGE_GROUP,VICTIM_SEX,VICTIM_RACE,STAT_MURDER_FLG
28912,318242320,318242320-1,18-24,MALE,BLACK,N
28913,319009737,319009737-2,25-44,MALE,WHITE,N
28914,320012929,320012929-1,25-44,MALE,WHITE HISPANIC,N
28915,320726813,320726813-1,<18,MALE,BLACK,N
28916,320989414,320989414-1,18-24,MALE,BLACK,N
...,...,...,...,...,...,...
29269,322313941,322313941-1,18-24,FEMALE,BLACK,N
29270,322454605,322454605-1,<18,MALE,BLACK,N
29271,322454605,322454605-2,<18,MALE,BLACK,N
29272,322608442,322608442-1,18-24,MALE,BLACK,Y


In [40]:
victims.drop_duplicates(subset=['INCIDENT_KEY', 'VICTIM_ID'], keep='first', inplace=True)
victims['VICTIM_ID'].value_counts()

VICTIM_ID
327260762-1        1
299453927-1        1
212336415-1        1
280768895-1        1
141824677-27687    1
                  ..
212640532-1        1
139621460-27183    1
138899428-26974    1
138899428-26973    1
305930898-1        1
Name: count, Length: 29128, dtype: int64

### Fix VICTIM_AGE_GROUP error

In [41]:
victims[victims['VICTIM_AGE_GROUP']=='1022']

,INCIDENT_KEY,VICTIM_ID,VICTIM_AGE_GROUP,VICTIM_SEX,VICTIM_RACE,STAT_MURDER_FLG
26463,248480012,248480012-1,1022,MALE,BLACK,N


In [42]:
victims['VICTIM_AGE_GROUP'] = victims['VICTIM_AGE_GROUP'].replace('1022', 'UNKNOWN')
victims['VICTIM_AGE_GROUP'].value_counts()

VICTIM_AGE_GROUP
25-44      13359
18-24      10491
<18         3016
45-64       1988
65+          214
UNKNOWN       60
Name: count, dtype: int64

# OFFENDERS DF 
#### - INCIDENT_KEY: unique ID for each incident, PK across dfs 
#### - PERP_ID: unique ID for each offender 
#### - PERP_AGE_GROUP: offender's age group 
#### - PERP_SEX: offender's sex description
#### - PERP_RACE: offender's race description 

In [43]:
offenders = pd.read_csv(r"C:\Users\arime\Downloads\Shooting_Offenders_(2006-Present)_20260920.csv")
offenders.shape

(19163, 5)

In [44]:
offenders.head()

,INCIDENT_KEY,PERP_ID,PERP_AGE_GROUP,PERP_SEX,PERP_RACE
0,288208647,288208647-1,25-44,MALE,ASIAN / PACIFIC ISLANDER
1,73517103,73517103-18843,25-44,MALE,ASIAN / PACIFIC ISLANDER
2,193462348,193462348-1,25-44,MALE,ASIAN / PACIFIC ISLANDER
3,186606271,186606271-1,18-24,MALE,ASIAN / PACIFIC ISLANDER
4,168757750,168757750-26618,25-44,MALE,ASIAN / PACIFIC ISLANDER


In [45]:
offenders.dtypes

INCIDENT_KEY       int64
PERP_ID           object
PERP_AGE_GROUP    object
PERP_SEX          object
PERP_RACE         object
dtype: object

In [46]:
offenders.isnull().sum()

INCIDENT_KEY       0
PERP_ID            0
PERP_AGE_GROUP    30
PERP_SEX           0
PERP_RACE          0
dtype: int64

In [47]:
offenders[offenders['PERP_AGE_GROUP'].isnull()]
#unknown offenders

,INCIDENT_KEY,PERP_ID,PERP_AGE_GROUP,PERP_SEX,PERP_RACE
17185,16895017,16895017-10196,NaN,UNKNOWN,UNKNOWN
17190,10454369,10454369-9353,NaN,UNKNOWN,UNKNOWN
17191,10879226,10879226-9605,NaN,UNKNOWN,UNKNOWN
17195,10333338,10333338-9305,NaN,UNKNOWN,UNKNOWN
17233,10333318,10333318-9282,NaN,UNKNOWN,UNKNOWN
17235,10454366,10454366-9349,NaN,UNKNOWN,UNKNOWN
17267,10137410,10137410-9109,NaN,UNKNOWN,UNKNOWN
17369,10333342,10333342-9310,NaN,UNKNOWN,UNKNOWN
17404,10879217,10879217-9588,NaN,UNKNOWN,UNKNOWN
17405,10804585,10804585-9536,NaN,UNKNOWN,UNKNOWN


### Change NaN to UNKNOWN

In [48]:
offenders['PERP_AGE_GROUP'] = offenders['PERP_AGE_GROUP'].fillna('UNKNOWN')

In [49]:
offenders.isnull().sum()

INCIDENT_KEY      0
PERP_ID           0
PERP_AGE_GROUP    0
PERP_SEX          0
PERP_RACE         0
dtype: int64

### Inspect categorical columns

In [50]:
#create df of the categorical columns, done manually to inspect the columns that will be relevant for my analysis 
categorical_columns = offenders[['PERP_ID','PERP_AGE_GROUP','PERP_SEX','PERP_RACE']]

#loop through every column of categorical_columns df, print their unique values and the counts of them
for column in categorical_columns: 
    print(f"Column: {column}")
    print(categorical_columns[column].value_counts())
    print("-" * 30)

Column: PERP_ID
PERP_ID
319753355-1        2
320708739-1        2
322341906-1        2
322629575-1        2
322477815-1        2
                  ..
190144578-2        1
164781178-26320    1
136393517-23534    1
219751446-1        1
325386857-1        1
Name: count, Length: 19047, dtype: int64
------------------------------
Column: PERP_AGE_GROUP
PERP_AGE_GROUP
18-24      6861
25-44      6260
UNKNOWN    3394
<18        1870
45-64       711
65+          61
1020          1
1028          1
2021          1
940           1
224           1
1822          1
Name: count, dtype: int64
------------------------------
Column: PERP_SEX
PERP_SEX
MALE       17299
UNKNOWN     1395
FEMALE       469
Name: count, dtype: int64
------------------------------
Column: PERP_RACE
PERP_RACE
BLACK                             12856
WHITE HISPANIC                     2628
UNKNOWN                            1749
BLACK HISPANIC                     1464
WHITE                               291
ASIAN / PACIFIC ISLANDER

### Drop duplicate PERP_IDs

In [51]:
offenders[offenders.duplicated(subset=['INCIDENT_KEY', 'PERP_ID'])]

,INCIDENT_KEY,PERP_ID,PERP_AGE_GROUP,PERP_SEX,PERP_RACE
6859,322341906,322341906-1,<18,MALE,BLACK
18902,320398603,320398603-1,18-24,MALE,WHITE HISPANIC
18903,320485441,320485441-1,25-44,MALE,BLACK
18904,321570558,321570558-1,25-44,MALE,BLACK
18905,320799697,320799697-1,25-44,MALE,WHITE HISPANIC
...,...,...,...,...,...
19153,319325648,319325648-1,45-64,MALE,BLACK HISPANIC
19154,321055998,321055998-1,18-24,MALE,WHITE HISPANIC
19155,322199412,322199412-1,18-24,MALE,BLACK
19156,322705412,322705412-1,18-24,FEMALE,BLACK


In [52]:
offenders.drop_duplicates(subset=['INCIDENT_KEY', 'PERP_ID'], keep='first', inplace=True)
offenders['PERP_ID'].value_counts()

PERP_ID
326622635-1       1
288208647-1       1
73517103-18843    1
193462348-1       1
186606271-1       1
                 ..
214102398-2       1
214102398-3       1
220063007-1       1
220553700-1       1
59801053-16524    1
Name: count, Length: 19047, dtype: int64

### Change PERP_AGE_GROUP values to UNKNOWN

In [53]:
offenders['PERP_AGE_GROUP'] = offenders['PERP_AGE_GROUP'].replace(['1020','1028','2021','940','224','1822'], 'UNKNOWN')
offenders['PERP_AGE_GROUP'].value_counts()

PERP_AGE_GROUP
18-24      6823
25-44      6222
UNKNOWN    3400
<18        1844
45-64       697
65+          61
Name: count, dtype: int64

### Check value count of INCIDENT_KEY - some have multiple offenders per incident. Check all INCIDENT_KEY are in shootings

In [54]:
print(offenders['INCIDENT_KEY'].duplicated().any())

True


In [55]:
offenders['INCIDENT_KEY'].value_counts()

INCIDENT_KEY
141562036    10
194915415    10
137962902    10
140088782     9
185015378     8
             ..
323453267     1
322627430     1
323826255     1
323453266     1
322740886     1
Name: count, Length: 14836, dtype: int64

In [56]:
offenders[offenders['INCIDENT_KEY']==194915415]

,INCIDENT_KEY,PERP_ID,PERP_AGE_GROUP,PERP_SEX,PERP_RACE
2957,194915415,194915415-10,18-24,MALE,BLACK
2958,194915415,194915415-9,25-44,MALE,BLACK
2959,194915415,194915415-8,25-44,MALE,BLACK
2960,194915415,194915415-7,25-44,MALE,BLACK
2961,194915415,194915415-6,25-44,MALE,BLACK
2962,194915415,194915415-5,18-24,MALE,BLACK
2963,194915415,194915415-4,25-44,MALE,BLACK
2964,194915415,194915415-3,25-44,MALE,BLACK
2965,194915415,194915415-2,25-44,MALE,BLACK
2966,194915415,194915415-1,25-44,MALE,BLACK


In [57]:
offenders[~offenders['INCIDENT_KEY'].isin(shootings['INCIDENT_KEY'])]

,INCIDENT_KEY,PERP_ID,PERP_AGE_GROUP,PERP_SEX,PERP_RACE
4372,78019135,78019135-19915,18-24,MALE,BLACK
13300,78019135,78019135-19916,18-24,MALE,BLACK HISPANIC


# MERGE DFS

In [58]:
final_shootings = pd.merge(shootings, victims, on='INCIDENT_KEY', how='left')

In [59]:
final_shootings = pd.merge(final_shootings, offenders, on='INCIDENT_KEY', how='left')

In [60]:
final_shootings.head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,TIME_OF_DAY,BORO,BORO_TOTAL,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,...,Longitude,VICTIM_ID,VICTIM_AGE_GROUP,VICTIM_SEX,VICTIM_RACE,STAT_MURDER_FLG,PERP_ID,PERP_AGE_GROUP,PERP_SEX,PERP_RACE
0,9953245,2006-01-01,0 days 02:00:00,NIGHT,BRONX,BRONX,OUTSIDE,48,0.0,STREET,...,-73.894610,9953245-8156,<18,MALE,BLACK,N,9953245-9037,18-24,MALE,BLACK
1,9953252,2006-01-01,0 days 02:22:00,NIGHT,MANHATTAN,MANHATTAN,OUTSIDE,28,0.0,STREET,...,-73.947403,9953252-8168,25-44,MALE,BLACK,Y,9953252-9046,25-44,MALE,BLACK
2,9953252,2006-01-01,0 days 02:22:00,NIGHT,MANHATTAN,MANHATTAN,OUTSIDE,28,0.0,STREET,...,-73.947403,9953252-8168,25-44,MALE,BLACK,Y,9953252-9051,25-44,MALE,BLACK
3,9953250,2006-01-01,0 days 02:34:00,NIGHT,QUEENS,QUEENS,OUTSIDE,114,0.0,COMMERCIAL,...,-73.933835,9953250-8165,25-44,MALE,BLACK,Y,9953250-9196,18-24,MALE,BLACK
4,9953250,2006-01-01,0 days 02:34:00,NIGHT,QUEENS,QUEENS,OUTSIDE,114,0.0,COMMERCIAL,...,-73.933835,9953250-8165,25-44,MALE,BLACK,Y,9953250-9044,25-44,MALE,BLACK


### Merge checks

In [61]:
print(shootings['INCIDENT_KEY'].nunique() == final_shootings['INCIDENT_KEY'].nunique())

True


In [62]:
print(final_shootings['OCCUR_DATE'].min())
print(final_shootings['OCCUR_DATE'].max())

2006-01-01 00:00:00
2026-06-30 00:00:00


In [63]:
final_shootings.isnull().sum()

INCIDENT_KEY              0
OCCUR_DATE                0
OCCUR_TIME                0
TIME_OF_DAY               0
BORO                      0
BORO_TOTAL                0
LOC_OF_OCCUR_DESC         0
PRECINCT                  0
JURISDICTION_CODE         2
LOC_CLASSFCTN_DESC        0
LOCATION_DESC             0
X_COORD_CD                0
Y_COORD_CD                0
Latitude                  0
Longitude                 0
VICTIM_ID                 0
VICTIM_AGE_GROUP          0
VICTIM_SEX                0
VICTIM_RACE               0
STAT_MURDER_FLG           0
PERP_ID               11159
PERP_AGE_GROUP        11159
PERP_SEX              11159
PERP_RACE             11159
dtype: int64

### Change new PERP nulls to UNKNOWN

In [64]:
new_perp_nulls = ['PERP_ID', 'PERP_AGE_GROUP', 'PERP_SEX', 'PERP_RACE']
final_shootings[new_perp_nulls] = final_shootings[new_perp_nulls].fillna('UNKNOWN')

print(final_shootings.isnull().sum())

INCIDENT_KEY          0
OCCUR_DATE            0
OCCUR_TIME            0
TIME_OF_DAY           0
BORO                  0
BORO_TOTAL            0
LOC_OF_OCCUR_DESC     0
PRECINCT              0
JURISDICTION_CODE     2
LOC_CLASSFCTN_DESC    0
LOCATION_DESC         0
X_COORD_CD            0
Y_COORD_CD            0
Latitude              0
Longitude             0
VICTIM_ID             0
VICTIM_AGE_GROUP      0
VICTIM_SEX            0
VICTIM_RACE           0
STAT_MURDER_FLG       0
PERP_ID               0
PERP_AGE_GROUP        0
PERP_SEX              0
PERP_RACE             0
dtype: int64


# Save as csv

In [65]:
final_shootings.to_csv('nyc_shootings.csv', index=False)